# DynLang-SLAM: Language Pipeline on Colab
Make sure you're using a **GPU runtime** (Runtime → Change runtime type → A100/T4)

In [ ]:
# Check GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print(f'torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')

In [ ]:
# Install dependencies
!pip install gsplat sam2 open_clip_torch omegaconf -q

In [ ]:
# Verify gsplat
from gsplat import rasterization
print('gsplat OK')

# Pre-download CLIP ViT-L/14 from OpenAI CDN (no HuggingFace auth needed)
import os, urllib.request
clip_cache = os.path.expanduser("~/.cache/clip")
os.makedirs(clip_cache, exist_ok=True)
clip_path = os.path.join(clip_cache, "ViT-L-14.pt")
if not os.path.exists(clip_path):
    print("Downloading CLIP ViT-L/14 from OpenAI CDN (~890MB)...")
    urllib.request.urlretrieve(
        "https://openaipublic.azureedge.net/clip/models/"
        "b8cca3fd41ae0c99ba7e8951adf17d267cdb84cd88be6f7c2e0eca1737a03836/ViT-L-14.pt",
        clip_path
    )
    print(f"CLIP weights saved to {clip_path}")
else:
    print(f"CLIP weights already cached at {clip_path}")
print(f"  Size: {os.path.getsize(clip_path)/1e6:.0f} MB")

In [ ]:
# Upload your project zip or clone from GitHub
# Option 1: Upload zip
from google.colab import files
# uploaded = files.upload()  # uncomment to upload

# Option 2: Clone from GitHub (if you pushed it)
# !git clone https://github.com/yourusername/DynLang-SLAM.git

# Option 3: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Copy project from Drive and extract
!cp /content/drive/MyDrive/DynLang-SLAM.zip /content/ 2>/dev/null || echo 'Upload DynLang-SLAM.zip to Google Drive root'
!cd /content && unzip -qo DynLang-SLAM.zip 2>/dev/null

# Auto-fix: Windows zips may nest the path — find and move to expected location
import glob, shutil, os
if not os.path.exists('/content/DynLang-SLAM/configs/default.yaml'):
    results = glob.glob('/content/**/dynlang_slam/__init__.py', recursive=True)
    if results:
        actual_root = os.path.dirname(os.path.dirname(results[0]))
        print(f"Found project at: {actual_root}")
        # Remove old target if exists (empty/broken)
        if os.path.exists('/content/DynLang-SLAM'):
            shutil.rmtree('/content/DynLang-SLAM')
        shutil.move(actual_root, '/content/DynLang-SLAM')
        print("Moved to /content/DynLang-SLAM")
    else:
        print("ERROR: Could not find project in zip!")
else:
    print("Project found at /content/DynLang-SLAM")

!ls /content/DynLang-SLAM/

In [ ]:
# Download SAM2 checkpoint
!mkdir -p /content/DynLang-SLAM/checkpoints
!wget -q -O /content/DynLang-SLAM/checkpoints/sam2.1_hiera_tiny.pt https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_tiny.pt
!ls -lh /content/DynLang-SLAM/checkpoints/

In [ ]:
# Set multi-scale in config (Colab has enough VRAM)
import yaml
with open('/content/DynLang-SLAM/configs/default.yaml', 'r') as f:
    cfg = yaml.safe_load(f)
cfg['language']['scales'] = ['whole', 'part', 'subpart']
with open('/content/DynLang-SLAM/configs/default.yaml', 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)
print('Config updated: multi-scale enabled')

In [ ]:
# Run standalone language test
%cd /content/DynLang-SLAM
!python scripts/test_language.py

In [ ]:
# Run full language SLAM test
!python scripts/test_language_slam.py

In [ ]:
# View results
from IPython.display import Image, display
import glob
for img in sorted(glob.glob('results/language_test/query_*_relevancy.jpg')):
    print(img.split('/')[-1])
    display(Image(img, width=600))